# AI-Based Behavioral IDS - Model Training

This notebook demonstrates training of all three model components:
1. LSTM Autoencoder
2. Isolation Forest
3. DNN Classifier
4. Hybrid Ensemble

## Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from data.data_loader import DataLoader
from data.preprocessor import DataPreprocessor
from data.smote_balancing import apply_smote_enn
from models.lstm_autoencoder import LSTMAutoencoder
from models.isolation_forest import IsolationForestIDS
from models.dnn_classifier import DNNClassifier
from models.hybrid_ids import HybridIDS

print("Libraries loaded!")

## 1. Prepare Data

In [ ]:
# Generate/load data
df = DataLoader.generate_synthetic_data(n_samples=5000, n_features=47)

# Preprocess
preprocessor = DataPreprocessor(sequence_length=100, use_pca=False)
results = preprocessor.fit_transform(df, test_size=0.2)

X_train_seq = results['X_train_seq']
X_test_seq = results['X_test_seq']
y_train = results['y_train_flat']
y_test = results['y_test_flat']
X_train_flat = results['X_train_flat']
X_test_flat = results['X_test_flat']
n_features = results['n_features']

print(f"Train sequences: {X_train_seq.shape}")
print(f"Test sequences: {X_test_seq.shape}")
print(f"Flat features: {X_train_flat.shape}")
print(f"Classes: {np.unique(y_train)}")

## 2. Train LSTM Autoencoder

In [ ]:
# Train on normal traffic only
normal_mask = y_train[:len(X_train_seq)] == 0
X_train_normal = X_train_seq[normal_mask]

print(f"Normal samples for LSTM training: {len(X_train_normal)}")

lstm = LSTMAutoencoder(
    sequence_length=100,
    n_features=n_features,
    latent_dim=16,
    lstm_units=[64, 32],
    dropout_rate=0.2
)

lstm.build_model()
lstm_history = lstm.train(
    X_train_normal, 
    epochs=20,  # Reduced for demo
    batch_size=64,
    patience=5
)

# Set threshold
lstm.compute_threshold(X_train_normal[:500], method='percentile', percentile=95)
print(f"\nLSTM threshold: {lstm.threshold:.6f}")

## 3. Train Isolation Forest

In [ ]:
# Use subset for speed
train_size = min(len(X_train_flat), 10000)
idx = np.random.choice(len(X_train_flat), train_size, replace=False)
X_if_train = X_train_flat[idx]

if_model = IsolationForestIDS(
    n_estimators=100,  # Reduced for demo
    max_samples=256,
    contamination=0.1
)

if_model.build_model()
if_model.train(X_if_train)

print("Isolation Forest trained!")

## 4. Train DNN Classifier

In [ ]:
# Get anomaly scores for DNN training
_, lstm_scores = lstm.predict_anomaly(X_train_seq[:len(X_train_flat)])
_, if_scores = if_model.predict(X_train_flat[:len(X_train_seq)])

# Normalize scores
lstm_scores_norm = (lstm_scores - lstm_scores.min()) / (lstm_scores.max() - lstm_scores.min() + 1e-10)
if_scores_norm = (if_scores - if_scores.min()) / (if_scores.max() - if_scores.min() + 1e-10)

# Combine features + scores
n_seq = min(len(X_train_flat), len(X_train_seq))
X_dnn = np.column_stack([X_train_flat[:n_seq], lstm_scores_norm, if_scores_norm])
y_dnn = y_train[:n_seq]

print(f"DNN input shape: {X_dnn.shape}")

dnn = DNNClassifier(
    n_features=X_dnn.shape[1],
    n_classes=2,
    hidden_layers=[128, 64, 32],
    dropout_rates=[0.3, 0.3, 0.2]
)

dnn.build_model()
dnn_history = dnn.train(X_dnn, y_dnn, epochs=20, batch_size=64, patience=5)

print("DNN Classifier trained!")

## 5. Hybrid Ensemble Evaluation

In [ ]:
# Create hybrid model
hybrid = HybridIDS(
    sequence_length=100,
    n_features=n_features,
    lstm_weight=0.4,
    if_weight=0.3,
    dnn_weight=0.3
)

# Assign trained sub-models
hybrid.lstm_ae = lstm
hybrid.isolation_forest = if_model
hybrid.dnn_classifier = dnn
hybrid.is_trained = True

# Evaluate
metrics = hybrid.evaluate(X_test_seq, X_test_flat, y_test)

# Print summary
print("\n" + "="*60)
print("FINAL RESULTS SUMMARY")
print("="*60)
for comp, m in metrics.items():
    print(f"\n{comp.upper()}:")
    for k, v in m.items():
        if isinstance(v, (int, float, np.integer, np.floating)):
            print(f"  {k}: {v:.4f}")

## 6. Visualization

In [ ]:
# Compare components
components = ['ensemble', 'lstm', 'isolation_forest', 'dnn']
accuracies = [metrics[c]['accuracy'] for c in components]
f1_scores = [metrics[c]['f1'] for c in components]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

x = range(len(components))
axes[0].bar(x, accuracies, color='steelblue', edgecolor='black')
axes[0].set_xticks(x)
axes[0].set_xticklabels([c.replace('_', ' ').title() for c in components], rotation=45)
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy Comparison', fontweight='bold')
axes[0].set_ylim(0.8, 1.0)

axes[1].bar(x, f1_scores, color='coral', edgecolor='black')
axes[1].set_xticks(x)
axes[1].set_xticklabels([c.replace('_', ' ').title() for c in components], rotation=45)
axes[1].set_ylabel('F1 Score')
axes[1].set_title('F1 Score Comparison', fontweight='bold')
axes[1].set_ylim(0.8, 1.0)

plt.tight_layout()
plt.show()

print("\nTraining complete!")